# Train a Hallucination Probe on Llama 3.1 8B

Notebook for Week 2 of the BlueDot research sprint. We'll:

1. Set up the environment (clone repo, install deps, HF login)
2. Train a **vanilla linear probe** on a small subset (200 examples) of LongFact annotations
3. Run inference with the probe we just trained on a few test examples

**Time**: ~1–1.5 hours total. Most of that is the training step (15–30 min on Colab Pro).

**Prerequisites**:
- **Colab Pro or pay-as-you-go** — free T4 doesn't have enough VRAM for 8B in bf16.
- **HuggingFace token** with read access (Llama 3.1 8B is gated). Request access at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct, then create a token at https://huggingface.co/settings/tokens.
- **Add the token as a Colab secret** named `HF_TOKEN` (sidebar → key icon).

**Goal**: see a probe successfully fire on hallucinated entities in held-out text. Not aiming to match the paper's numbers — just validating that the full pipeline works end-to-end on a small subset.

## 1. Clone the repo

We'll clone the codebase and `cd` into it. The script entry point (`python -m probe.train`) is structured as a Python module under this directory.

**Replace `REPO_URL` below** with either:
- The upstream repo URL, or
- Your own fork's URL (recommended — you'll likely want to make changes).

If you haven't pushed your local clone to GitHub yet, you can do it in one command from your local terminal: `gh repo create <name> --public --source=. --push`.

In [ ]:
REPO_URL = "https://github.com/<YOUR_USERNAME>/hallucination_probes.git"  # ← REPLACE

!git clone {REPO_URL}
%cd hallucination_probes

## 2. Install dependencies

Locally we use `uv`, but in Colab plain `pip` is simpler. The `-e .` flag installs the package in editable mode using `pyproject.toml`, so you can edit source files in this Colab session and they take effect on the next import.

This installs torch, transformers, peft, datasets, and the local `probe`/`utils` packages. Takes 2–4 minutes.

In [ ]:
!pip install -q -e .

## 3. HuggingFace authentication

Llama 3.1 8B is gated, so we need an authenticated session to download it. The cell below reads the `HF_TOKEN` Colab secret and logs in.

If you haven't set the secret yet: open the **🔑 sidebar** in Colab, add a secret named `HF_TOKEN`, paste your token, toggle **Notebook access**.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("Logged in via Colab secret.")
except Exception as e:
    print(f"Couldn't read Colab secret ({e}). Falling back to interactive login.")
    login()  # interactive — paste token when prompted

## 4. Disable W&B logging (optional)

The training script logs to Weights & Biases by default. For this small one-off run we'll disable it. (If you want to use W&B for nicer loss curves, comment this cell out and run `!wandb login` instead.)

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

## 5. Verify GPU

What hardware did Colab give us? On Pro you'll typically see **V100 (16 GB)**, **L4 (24 GB)**, or sometimes **A100 (40 GB)**. Llama 3.1 8B in bf16 is ~16 GB, so V100 is tight — if you got V100 and OOM during training, lower `per_device_train_batch_size` to 1 (already the default in our config).

If you got a T4 (15 GB), training will likely OOM. Either reconnect (Runtime → Change runtime type) for a better GPU, or modify the training to use 8-bit quantization.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 6. Write our custom training config

The repo's default `configs/train_config.yaml` trains across **8 datasets** spanning **5 different base models** — way too much for our quick test. We'll write our own minimal config: one dataset, 200 examples, 1 epoch, vanilla linear probe (no LoRA, no regularizers).

**Each parameter has a comment.** Read through them — these are the knobs you'll be tweaking in Week 3 when adapting for disempowerment. The most interesting to mess with for ablations:

- `pos_weight` / `neg_weight` — try setting `pos_weight: 1.0` and watch the probe collapse to predicting all zeros.
- `layer` — try `15` (mid) or `5` (early) instead of `30` (penultimate). See where the hallucination signal lives in the model.
- `default_ignore` — try `true` and see how the loss landscape changes when only annotated tokens are supervised.
- `max_num_samples` — bump to 500 or remove entirely if you have time for a longer run.

In [ ]:
config_yaml = """
# ============================================================================
# my_small_config.yaml — Vanilla Linear Hallucination Probe
# ============================================================================
# Trains one linear probe on a 200-example subset of LongFact annotations,
# targeting Llama 3.1 8B at layer 30 (penultimate). No LoRA, no regularizers
# — just BCE loss with class weighting.
# ============================================================================

probe_config:
  probe_id: \"my_first_probe_linear_8b\"
  # ^ Name of your probe. Used as the output directory name
  #   (value_head_probes/{probe_id}/) and HF upload name if upload_to_hf=true.

  model_name: \"meta-llama/Meta-Llama-3.1-8B-Instruct\"
  # ^ Base model. MUST match the model that produced the annotations in the
  #   dataset (otherwise the activations are from a different distribution).

  layer: 30
  # ^ Which transformer block's hidden states to tap. Llama 3.1 8B has 32
  #   layers; 30 = penultimate. The paper finds penultimate works best for
  #   hallucination detection. Try 15 (mid) or 5 (early) for layer ablation.

  # === LoRA (DISABLED for the vanilla linear probe) ===
  lora_layers: null
  # ^ null = no LoRA. The base model stays frozen; only the linear head trains.
  #   Set to \"all\" or a layer list to enable LoRA-augmented probes.
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  # ^ Above three only matter if lora_layers != null.

  load_from: null
  # ^ null = train from scratch (random-init linear head).
  #   \"hf\" = download a pretrained probe from hf_repo_id and resume.
  #   \"disk\" = load from a local probe directory.

  hf_repo_id: \"obalcells/hallucination-probes\"
  # ^ HF repo to load from (if load_from=\"hf\") or push to (if upload_to_hf).
  #   Not used in this config since we train from scratch and don't upload.

  threshold: 0.5
  # ^ Decision threshold for converting probe probabilities to binary
  #   predictions during evaluation (used for thresholded F1).


# === Logging / saving ===
wandb_project: \"hallucination-probes\"
# ^ W&B project name. We've set WANDB_DISABLED=true, so this is ignored.
upload_to_hf: false
# ^ Push the trained probe to HF Hub? Requires HF_WRITE_TOKEN if true.
save_evaluation_metrics: true
save_roc_curves: false
dump_raw_eval_results: false


# === Training hyperparameters ===
per_device_train_batch_size: 1
# ^ How many examples per gradient step. Small to fit on 16 GB GPUs.
per_device_eval_batch_size: 1

high_loss_threshold: null
# ^ If set, tokens whose LM perplexity exceeds this get masked out of
#   training. null = off.

lambda_lm: 0.0
# ^ Weight on the LM regularization loss (cross-entropy on next-token
#   prediction, to keep base model from drifting). 0 = no LM penalty.
#   Only relevant for LoRA probes.

lambda_kl: 0.0
# ^ Weight on the KL regularization loss (KL divergence between LoRA-modified
#   model and frozen base). Like lambda_lm, irrelevant for vanilla.

anneal_max_aggr: true
# ^ Whether to anneal the max-aggregation loss term. (See loss.py — it's a
#   span-level loss alternative to per-token BCE, kept for compatibility.)
anneal_warmup: 1.0

probe_head_lr: 1e-3
# ^ Learning rate for the linear head. The only thing that learns in a
#   vanilla probe.

lora_lr: 1e-4
# ^ Learning rate for LoRA parameters. Irrelevant since lora_layers=null.

enable_gradient_checkpointing: true
# ^ Trades compute for memory by re-computing activations during backward
#   instead of storing them. Slower but lets you fit larger models / batches.

gradient_accumulation_steps: 4
# ^ Number of forward/backward passes per optimizer step. Effective batch
#   size = per_device_train_batch_size * gradient_accumulation_steps = 4.

max_grad_norm: 1.0
# ^ Gradient clipping threshold. Standard.

logging_steps: 5
# ^ Print loss every N steps.

seed: 42


# === Training datasets ===
# Just one dataset, small subset, to keep the run fast.
train_datasets:
  - dataset_id: \"longfact_8b_train_small\"
    hf_repo: \"obalcells/longfact-annotations\"
    subset: \"Meta-Llama-3.1-8B-Instruct\"
    split: \"train\"
    max_length: 1536
    # ^ Truncate examples beyond this many tokens. Median 8B generation is
    #   ~1230 tokens, so this fits most.

    default_ignore: false
    # ^ false = non-span assistant tokens get label 0.0 (cheap negative
    #   supervision). true = mask them out. False is the longform default.

    last_span_token: false
    # ^ false = label every token in a span. true = label only the last.

    ignore_buffer: 0
    # ^ Mask out N tokens around each *positive* span (asymmetric — only
    #   buffers around hallucinations, not supported entities). 0 = off.

    pos_weight: 10.0
    neg_weight: 10.0
    # ^ Per-token loss multipliers. 10x the cheap (1.0) negatives. This
    #   counterbalances class imbalance — try pos_weight=1.0 to see the
    #   probe collapse to predicting all zeros (good ablation).

    shuffle: true
    seed: 42
    process_on_the_fly: false

    max_num_samples: 200
    # ^ Take only first 200 (after shuffle). Train split has 510 total;
    #   200 keeps things fast.


# === Evaluation datasets ===
eval_datasets:
  - dataset_id: \"longfact_8b_test_small\"
    hf_repo: \"obalcells/longfact-annotations\"
    subset: \"Meta-Llama-3.1-8B-Instruct\"
    split: \"test\"
    max_length: 1536
    pos_weight: 10.0
    neg_weight: 10.0
    default_ignore: false
    shuffle: false
    max_num_samples: 50
    # ^ Eval on 50 test examples for quick feedback.
"""

with open("my_small_config.yaml", "w") as f:
    f.write(config_yaml)

print("Wrote my_small_config.yaml")
print(f"Length: {len(config_yaml)} chars")

## 7. Run training

Single command. The script will:

1. **Download Llama 3.1 8B** (~16 GB; takes 5–10 min on Colab's network).
2. **Download the annotations dataset** from HF (small).
3. **Tokenize and label** all 200 training + 50 eval examples — see the per-token labels and weights you read about in Phase 3.
4. **Freeze the base model** (`requires_grad=False` on every base param).
5. **Initialize a fresh linear head** (`nn.Linear(hidden_size=4096, 1)`).
6. **Iterate**: forward through frozen base → hooked hidden states from layer 30 → linear head → BCE loss → backward → optimizer step. Only the linear head's weights update.
7. **Save** the probe to `value_head_probes/my_first_probe_linear_8b/`.

Expect ~15–30 min depending on your GPU. You'll see the loss decreasing in the printouts every 5 steps.

In [ ]:
!python -m probe.train --config my_small_config.yaml

## 8. Inspect the saved probe

Where did everything go? The probe directory holds:
- `probe_head.bin` — the trained linear weights (`weight: [4096, 1]`, `bias: [1]`)
- `probe_config.json` — metadata (which layer, hidden size, etc.) so we can reload it

In [ ]:
import json
from pathlib import Path

probe_dir = Path("value_head_probes/my_first_probe_linear_8b")
print(f"Files in {probe_dir}:")
for f in sorted(probe_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

print("\nprobe_config.json contents:")
with open(probe_dir / "probe_config.json") as f:
    print(json.dumps(json.load(f), indent=2))

## 9. Reload model + load our trained probe for inference

The training process exited, so the model is gone from memory. We'll reload Llama 3.1 8B and attach our freshly trained linear head.

(In Week 3 you might write a notebook that does training + inference in one process — but doing it this way is closer to the production setup, where training runs separately from serving.)

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from probe.value_head_probe import ValueHeadProbe

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PROBE_PATH = Path("value_head_probes/my_first_probe_linear_8b")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading base model (this takes a couple minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# Freeze (defensive — we're only doing inference)
for p in model.parameters():
    p.requires_grad = False

print("Loading our trained probe...")
probe = ValueHeadProbe(model=model, path=PROBE_PATH)
probe.eval()

print(f"\nProbe ready. Layer={probe.layer_idx}, hidden_size={probe.value_head.in_features}")

## 10. Run the probe on test examples

Now the moment of truth: load a few held-out test examples, run them through `model + probe`, and look at the per-span scores.

For each annotated span we'll print:
- The span text
- The ground-truth label (Supported / Not Supported / Insufficient Information)
- The probe's average sigmoid score on that span

**What to expect**: "Not Supported" spans should have *higher* scores than "Supported" spans. With only 200 training examples, expect noisy results — pretrained probes (trained on many thousands of examples) would score better. The point is to see the gradient is in the right direction.

In [ ]:
from datasets import load_dataset
from utils.tokenization import find_string_in_tokens

ds = load_dataset("obalcells/longfact-annotations", "Meta-Llama-3.1-8B-Instruct", split="test")

for idx in [0, 1, 2]:
    ex = ds[idx]
    user_msg = ex['conversation'][0]['content']
    print(f"\n{'='*70}")
    print(f"Example {idx}: {user_msg[:80]}...")
    print('='*70)

    # Apply chat template + tokenize
    text = tokenizer.apply_chat_template(ex['conversation'], tokenize=False)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1536).to(model.device)

    # Forward through model + probe
    with torch.no_grad():
        out = probe(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
    probe_probs = torch.sigmoid(out['probe_logits']).squeeze(-1).squeeze(0).cpu().numpy()

    # Score each annotation
    print(f"{'Span':<45} {'Label':<28} {'Score':>6}")
    print('-' * 80)
    for ann in ex['annotations'][:8]:
        if ann is None or 'index' not in ann or not isinstance(ann.get('index'), int):
            continue
        try:
            tok_slice = find_string_in_tokens(ann['span'], inputs.input_ids[0], tokenizer)
            avg_score = probe_probs[tok_slice].mean()
            print(f"{ann['span'][:43]:<45} {str(ann['label']):<28} {avg_score:>6.3f}")
        except Exception as e:
            print(f"{ann['span'][:43]:<45} {str(ann['label']):<28} (error: {e})")

## 11. Color-coded per-token visualization

More vivid view: print the assistant's response token-by-token, colored by the probe's score. **Red = high score (probe thinks hallucination)**, **yellow = uncertain**, **plain = low score**.

Watch where the probe lights up — does it correlate with the actual hallucinated entities?

In [ ]:
from termcolor import colored

ex = ds[0]
text = tokenizer.apply_chat_template(ex['conversation'], tokenize=False)
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1536).to(model.device)

with torch.no_grad():
    out = probe(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
probe_probs = torch.sigmoid(out['probe_logits']).squeeze().cpu().numpy()

# Find where the assistant's response starts (skip the prompt + chat template)
from utils.tokenization import find_assistant_tokens_slice
input_str = tokenizer.decode(inputs.input_ids[0])
asst_slice = find_assistant_tokens_slice(inputs.input_ids[0], input_str, tokenizer)
start_idx = asst_slice.stop

# Print color-coded
print(f"Assistant response (probe score per token):\n")
for i, tok_id in enumerate(inputs.input_ids[0][start_idx:], start=start_idx):
    tok_str = tokenizer.decode([tok_id])
    p = probe_probs[i]
    if p > 0.5:
        print(colored(tok_str, 'red', attrs=['bold']), end='')
    elif p > 0.3:
        print(colored(tok_str, 'yellow'), end='')
    else:
        print(tok_str, end='')
print()

# Also print the ground truth annotations for comparison
print(f"\n\n{'='*70}\nGround-truth annotations (for comparison):")
for ann in ex['annotations'][:10]:
    if ann is None or 'span' not in ann:
        continue
    print(f"  [{ann.get('label', '?'):>30}]  {ann.get('span', '')}")

## 12. What to try next

Some quick experiments you can run by editing `my_small_config.yaml` and re-running training. Each takes ~15–30 min on Pro.

**Class-weight ablation** (most pedagogical):
- Set `pos_weight: 1.0` in the train_datasets section.
- Retrain. Observe that the probe collapses to predicting near-zero on everything.
- Why: with rare positives and equal weighting, BCE is minimized by predicting the majority class (negative) everywhere. The 10x weighting was load-bearing.

**Layer ablation** (where does the signal live?):
- Change `layer: 30` to `layer: 15` (mid) and retrain.
- Try `layer: 5` (early) too.
- Compare per-span scores. The paper finds penultimate works best — does your tiny experiment agree?

**Sample size**:
- Bump `max_num_samples: 200` to `500` or remove the field entirely (uses all 510 train examples).
- Does the probe get noticeably better?

**`default_ignore` flip**:
- Set `default_ignore: true` in the train_datasets section.
- Now only annotated tokens get supervised — everything else is masked out.
- Compare to the default. The probe is now learning "hallucinated entity vs supported entity" rather than "hallucinated entity vs everything else."

## Bridge to Week 3

What you'll reuse for the disempowerment adaptation:
- This same training entry point (`python -m probe.train --config ...`).
- The dataset format (`conversation` + `annotations` with `index`/`span`/`label`).
- The vanilla linear probe setup (no LoRA).

What you'll need to swap:
- The `hf_repo` in train_datasets — point at your own dataset of disempowerment-labeled conversations.
- The label vocabulary in `_MAP_LABEL_TO_SCALAR` (`probe/dataset_converters.py`) — Sharma-taxonomy categories instead of Supported/Not Supported.
- Possibly `last_span_token` and `default_ignore` — disempowerment may want different semantics around what counts as a labeled token.
- The annotation pipeline itself (`annotation_pipeline/`) — your own LLM-judge prompt for the Sharma taxonomy.